In [4]:
import pandas as pd
import numpy as np
from numpy import mean
from numpy import std
from numpy import dstack
from pandas import read_csv
from numpy import hstack
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

pathname = r'/Users/aya/Local Documents/NYU/Dissertation/data/BIOWIN_database/1_Datasets/Monthly/normal-28days.csv'
tagged_pathname = r'/Users/aya/Local Documents/NYU/Dissertation/data/BIOWIN_database/1_Datasets/Preprocessed/Ff-28days-final.csv'
tagged_csv = pd.read_csv(tagged_pathname)
in_csv = pd.read_csv(pathname)
len(in_csv.columns)

tagged_csv[30:50]
for i in in_csv.columns[15:]:
    print(i)
#for i in in_csv.columns[0:14]:
#    print(i)

def data_plotter(x, y, c, ylabel, title):
    fig_size = plt.rcParams["figure.figsize"]
    fig_size[0] = 15
    fig_size[1] = 10
    plt.rcParams["figure.figsize"] = fig_size

    fig, ax1 = plt.subplots()

    ax1.plot(x, y, marker = 'o', color = c, linestyle = '--')

    ax1.set_ylabel(ylabel)
    ax1.set_title(title)
    plt.scatter(y=y, x=x)

# Create the list of training variables used for training the LSTM to predict the next series of the risk sequence 
Xlist = list(in_csv.columns[15:])
#Xlist.append('Final Effluent Total suspended solids')
Xlist

# Append the Frequency tag to the training set of the target variable for risk prediction, TSS mg/L
df_X = pd.DataFrame()
for i in Xlist:
    df_X = df_X.append(in_csv[i])
df_X = df_X.append(tagged_csv['tagF'])

df_X = df_X.T

df_X.head()
DF_X = pd.DataFrame()
for i in Xlist:
    temparr = np.asarray(df_X[i])
    temparr = temparr.reshape(-1, 1)
    scaler = MinMaxScaler()
    MMtemparr = scaler.fit_transform(temparr)
    DF_X["MM!__" + i] = MMtemparr.flatten().tolist()
DF_X = DF_X.T
DF_X = DF_X.append(df_X['tagF'])
DF_X = DF_X.T

dataset = DF_X.to_numpy()

DF_X

def split_sequences(sequences, n_steps_in, n_steps_out):
  X, y = list(), list()
  for i in range(len(sequences)):
    # find the end of this pattern
    end_ix = i + n_steps_in
    out_end_ix = end_ix + n_steps_out + 1
    # check if we are beyond the dataset
    if out_end_ix > len(sequences):
        break
    # gather input and output parts of the pattern
    seq_x, seq_y = sequences[i:end_ix, :-1], sequences[end_ix-1:out_end_ix, -1]
    X.append(seq_x)
    y.append(seq_y)
  return np.array(X), np.array(y)



PST x3 COD - Total
PST x3 BOD - Total Carbonaceous
PST x3 N - Total Kjeldahl Nitrogen
PST x3 P - Total P
PST x3 pH
PST x3 Total suspended solids
PST x3 Volatile suspended solids
PST x3 Element HRT
PST x3 Gas - Dissolved oxygen
PST x3 P - Soluble phosphate
PST x3 N - Ammonia
PST x3 N - Nitrate
PST x3 N - Nitrite
PST x3 Alkalinity
PST x3 Flow
ML Recycle COD - Total
ML Recycle BOD - Total Carbonaceous
ML Recycle N - Total Kjeldahl Nitrogen
ML Recycle N - Total Kjeldahl Nitrogen (U)
ML Recycle P - Total P
ML Recycle pH
ML Recycle Total suspended solids
ML Recycle Volatile suspended solids
ML Recycle Gas - Dissolved oxygen
ML Recycle P - Soluble phosphate
ML Recycle N - Ammonia
ML Recycle N - Ammonia (U)
ML Recycle N - Nitrate
ML Recycle N - Nitrite
ML Recycle Alkalinity
ML Recycle Flow
ML Recycle Flow (S)
FST x5 COD - Total
FST x5 BOD - Total Carbonaceous
FST x5 N - Total Kjeldahl Nitrogen
FST x5 P - Total P
FST x5 pH
FST x5 Total suspended solids
FST x5 Total suspended solids (U)
FST x5 V

In [6]:
n_steps_in, n_steps_out = 3, 3
X, y = split_sequences(dataset,n_steps_in, n_steps_out)
print(X.shape, y.shape)
for i in range(len(X)):
    print(X[i], y[i])

(667, 3, 78) (667, 5)
[[0.55865436 0.57403081 0.29499149 0.03464709 0.         0.61150848
  0.55943743 0.1074486  0.09673173 0.         0.21696078 0.64812662
  0.         0.         0.67666126 0.95455518 0.973628   0.98868094
  0.98868094 0.99782387 0.         0.98858838 0.94310369 0.50937494
  0.11223959 0.08984525 0.08984525 0.84893437 0.1381019  0.12330515
  0.67666126 0.         0.76135861 1.         0.07589196 0.1102938
  0.         0.21312627 1.         0.15864449 0.20326828 1.
  0.06373696 0.04727558 0.9925458  0.10754631 0.         0.67666126
  0.94652741 0.94036388 0.97596984 0.97257423 0.         0.96766556
  0.9357075  0.24408638 0.62107448 0.06800757 0.04055451 1.
  0.06789934 0.         0.67666126 1.         1.         0.93104126
  0.9364456  0.         0.9343505  0.96655068 0.24408259 0.
  1.         0.         0.04136169 0.         0.         0.67666126]
 [0.41687826 0.42989847 0.19387143 0.0911524  0.0082999  0.51924429
  0.42448498 0.18951112 0.09590288 0.06808856 0.13

In [ ]:

# choose a number of time steps
n_steps_in, n_steps_out = 3, 3
X, y = split_sequences(dataset,n_steps_in, n_steps_out)
print(X.shape, y.shape)
for i in range(len(X)):
    print(X[i], y[i])

# flatten input: calculate the length of the input vector as the number of time steps mutilplied by the number of features or time series.
n_input = X.shape[1] * X.shape[2]
# use this vector size to reshape the input.
X = X.reshape((X.shape[0], n_input))
X.shape
def evaluate_model(trainX, trainy, testX, testy):
    verbose, epochs, batch_size = 0, 10, 32
    n_timesteps, n_features, n_outputs = trainX.shape[1], trainX.shape[2], trainy.shape[1]
    model = keras.Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(n_timesteps,n_features)))
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu')) model.add(Dropout(0.5))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(n_outputs, activation='softmax')) model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


###########
# define model with a multivariate input where the vector length is the used for the input dimension arguement of the model
model = keras.Sequential()
model.add(Dense(100, activation='relu', input_dim=n_input))
model.add(Dense(n_steps_out+2))
model.compile(optimizer='adam', loss='mse')
# fit model
model.fit(X, y, epochs=2000, verbose=0)


In [ ]:
# demonstrate prediction
print(len(X))
#x_input = X[51:150]
x_input = X[0:1]
x_input.shape
yhat = model.predict(x_input, verbose=0)

Y_hat = list()
for i in range(0,len(yhat)):
    Y_hat.append(yhat[i][0])
np.array(Y_hat)

m1 = yhat.flatten()
m1 = np.interp(np.arange(0, len(m1), 2), np.arange(0, len(m1)), m1)
len(m1)
m1
m1.shape[0]
m1
#m2 = DF_X['MM!__Final Effluent Total suspended solids'][0:3]
m2 = DF_X['tagF'][2:101]

m2
N = len(m1)
N2 = len(m2)
N2
len(Y_hat)
xYhat = np.linspace(0, len(Y_hat[2:101]), N, endpoint=True)
x1 = np.linspace(0, len(m1), N, endpoint=True)
x2 = np.linspace(0, len(m2), N2, endpoint=True)

fig, ax1 = plt.subplots()

ax1.plot(xYhat, Y_hat[2:101], marker = 'o', color = 'orange', linestyle = '--')
ax1.plot(x2, m2, marker = 'o', color = 'blue', linestyle = '--')

ax1.set_ylabel('yhat')
ax1.set_title('yhat (orange) vs y (blue)')